# 实验三：QQ邮箱采集（Playwright 版）— 已补全代码- 班级：数据1241　姓名：李锐　学号：112404260119- 四个阶段写在同一个 Notebook：登录 → 采集10封 → 滚动去重 → 详情正文- 课程环境：先双击 `start-course-visual.bat`，再在 JupyterLab 中打开本文件。- 本文件已补全指导书单元 1—13 的全部 `____`。**选择器需要按你自己的页面在开发者工具中核对**（见单元 3 的自检单元）。> 隐私提醒：`private_data/` 只放本机，不上传；提交前用 `data/` 中的脱敏 CSV 和 `images/` 中的脱敏截图。

## 单元1：导入工具，准备浏览器（第1—2课时）`start()` 启动 Playwright 工具，`launch()` 启动浏览器，`new_context()` 创建独立环境，`new_page()` 新建页面。`headless=False` 才能看到可视化浏览器画面，便于人工扫码。

In [ ]:
import sysfrom pathlib import Pathfrom uuid import uuid4import pandas as pdfrom playwright.async_api import async_playwright# 三个阶段共用的本机私有目录（不要上传）。PRIVATE = Path.cwd() / "private_data"PRIVATE.mkdir(exist_ok=True)# 提交用的脱敏数据与截图目录。PUBLIC = Path.cwd() / "data"PUBLIC.mkdir(exist_ok=True)SHOTS = Path.cwd() / "images"SHOTS.mkdir(exist_ok=True)# course_visual 属于课程已安装的 visual 环境，无需另发文件。# 本实验要求 lab03 与 L07_L08_visual 在 workspace 下同级。sys.path.insert(0, str(Path.cwd().parent / "L07_L08_visual"))import course_visualcourse_visual.prepare()p = await async_playwright().start()browser = await p.chromium.launch(    headless=False,    args=["--disable-dev-shm-usage"],)context = await browser.new_context(    viewport={"width": 1000, "height": 620}, accept_downloads=True)page = await context.new_page()page.set_default_timeout(20000)  # 20秒，毫秒单位，不需要 awaitprint("浏览器与页面已就绪")

## 单元2：打开邮箱，等待扫码并进入收件箱（第1—2课时）"页面已打开" 只说明 `goto()` 返回；"已经登录" 要靠**收件箱控件出现且可点击**来判断。这里最长等待 600000 毫秒（10 分钟），出现后立即继续，不用固定 `sleep`。

In [ ]:
await page.goto("https://mail.qq.com/", wait_until="domcontentloaded")print("请在浏览器画面中扫码", flush=True)# “收件箱”左侧区域选择器：请在你自己的页面中用开发者工具核对。# 若实际页面结构不同，把下面的 ".sidebar" 换成真实的左侧导航容器选择器。inbox = page.locator(".sidebar").get_by_text("收件箱", exact=True)await inbox.wait_for(timeout=600000)print("登录成功", flush=True)await inbox.click()print("已进入收件箱", flush=True)# 截图证据（提交前请遮挡账号、头像、邮件标题等私人信息）。await page.screenshot(path=str(SHOTS / "01_login_inbox_raw.png"))

## 单元3：自己找选择器（第3—4课时）指导书只给了两条线索，其余按页面实际结构在开发者工具中查找。下面填写的是新版 QQ 邮箱常见的写法；**不同账号/版本可能不同，运行单元 3 自检后按结果调整**。| 内容 | 本 Notebook 使用 | 说明 ||---|---|---|| 一封邮件 | `[data-mailid]` | 用属性定位，避免依赖易变 class || 发件人姓名 | `.cmp-account-nick` | 指导书给定 || 广告聚合入口 | `.gg-unread-count` | 指导书给定，应跳过 || 主题 | `.subject` | 按页面核对 || 时间 | `.time` | 按页面核对 || 发件人悬停区域 | `.cmp-account-nick` | 与发件人同一区域 || 卡片中的邮箱 | `.email-card-email` | 悬停后出现在整页，按页面核对 || 滚动容器 | `.mail-list-page .ui-float-scroll-body` | 指导书给出了内部滚动态类名 |

In [ ]:
ROW_CSS = "[data-mailid]"              # 一条邮件项目（用属性定位最稳定）SENDER_CSS = ".cmp-account-nick"        # 指导书给定AD_CSS = ".gg-unread-count"             # 指导书给定，广告聚合入口SUBJECT_CSS = ".subject"                # 行内主题TIME_CSS = ".time"                      # 行内时间HOVER_CSS = ".cmp-account-nick"         # 行内发件人悬停区域EMAIL_CSS = ".email-card-email"         # 整页中的邮箱卡片字段SCROLL_CSS = ".mail-list-page .ui-float-scroll-body"

In [ ]:
# 单元3自检：打印每个选择器匹配到的元素个数。# 若某项为 0，说明该选择器与你的页面不符，请到开发者工具中改正上面的常量。for name, css in [    ("ROW_CSS", ROW_CSS), ("SENDER_CSS", SENDER_CSS), ("AD_CSS", AD_CSS),    ("SUBJECT_CSS", SUBJECT_CSS), ("TIME_CSS", TIME_CSS),    ("HOVER_CSS", HOVER_CSS), ("EMAIL_CSS", EMAIL_CSS),    ("SCROLL_CSS", SCROLL_CSS),]:    print(f"{name:12s} {css:45s} -> {await page.locator(css).count()}")

## 单元4：定义列表解析函数（第3—4课时）`get_ids()` 一次性读取整批 ID，不保存旧元素对象，列表更新后不会拿到失效节点。`read_list_mail()` 先跳过广告入口，再读字段；读取邮箱前先移开鼠标并等待旧卡片消失，随后把发件人滚入可见区域、滚动停止后再悬停，保证读到的是**当前这封**邮件的邮箱。

In [ ]:
async def get_ids():    await page.locator(ROW_CSS).first.wait_for()    # 一次读取整批ID，避免列表更新时逐个保存旧元素。    ids = await page.locator(ROW_CSS).evaluate_all(        "rows => rows.map(row => row.getAttribute('data-mailid'))"    )    if not ids or not all(ids):        raise ValueError("没有取得有效邮件ID，请检查页面")    return idsdef get_row(mail_id):    return page.locator(f'{ROW_CSS}[data-mailid="{mail_id}"]')async def read_list_mail(mail_id):    row = get_row(mail_id)    # count() 为 0 表示没找到广告标记，是正常的普通邮件。    if await row.locator(AD_CSS).count() > 0:        return None    sender = await row.locator(SENDER_CSS).inner_text()    subject = await row.locator(SUBJECT_CSS).inner_text()    time_text = await row.locator(TIME_CSS).inner_text()    # 先移开鼠标，等待上一张卡片消失。    await row.locator(TIME_CSS).hover()    await page.locator(EMAIL_CSS).wait_for(state="hidden")    # 先把发件人滚入可见区域；滚动时页面会暂时停止响应悬停。    await row.locator(HOVER_CSS).scroll_into_view_if_needed()    await page.locator(".mail-list-page .ui-float-scroll-body-scrolling").wait_for(state="hidden")    # 滚动结束后，再悬停本封发件人，等待卡片显示。    await row.locator(HOVER_CSS).hover()    await page.locator(EMAIL_CSS).wait_for(state="visible")    email = await page.locator(EMAIL_CSS).inner_text()    return {        "邮件ID": mail_id,        "发件人": sender.strip(),        "邮箱": email.strip(),        "主题": subject.strip(),        "时间": time_text.strip(),    }# 先只解析一封，核对无误后再运行单元5。first_id = (await get_ids())[0]print(await read_list_mail(first_id))

## 单元5：采集10封（第3—4课时）用 `append()` 逐条加入列表，凑够 10 条普通邮件即停止。

In [ ]:
basic_records = []for mail_id in await get_ids():    record = await read_list_mail(mail_id)    if record is None:        continue    basic_records.append(record)    if len(basic_records) == 10:        breakprint("成功采集：", len(basic_records))

## 单元6：用 pandas 保存和回读（第3—4课时）`DataFrame()` 建表，`to_csv()` 保存，`index=False` 不写行号，`utf-8-sig` 便于 Excel 打开。后面三个阶段复用下面两个函数，不再另写一套。

In [ ]:
def save_records(records, filename, columns):    df = pd.DataFrame(records, columns=columns)    path = PRIVATE / filename    df.to_csv(path, index=False, encoding="utf-8-sig")    return pathdef read_records(path):    return pd.read_csv(path, dtype=str, keep_default_na=False)def save_public(records, filename, columns):    """生成脱敏CSV：邮件ID保留用于三份CSV之间的对应，其余字段换成代号。"""    rows = []    for i, r in enumerate(records, 1):        item = dict(r)        if "发件人" in item:            item["发件人"] = f"发件人{i:02d}"        if "邮箱" in item:            item["邮箱"] = f"sender{i:02d}@example.com"        if "主题" in item:            item["主题"] = f"主题样本{i:02d}"        if "正文" in item:            item["正文"] = f"[已脱敏正文，共{len(item['正文'])}字符]"        rows.append(item)    return save_records(rows, filename, columns)BASIC_COLUMNS = ["发件人", "邮箱", "主题", "时间"]basic_path = save_records(basic_records, "mail_basic.csv", BASIC_COLUMNS)basic_df = read_records(basic_path)print("CSV记录数：", len(basic_df))basic_df.head(10)# 生成提交用的脱敏CSV。public_basic_path = save_public(basic_records, "mail_basic_public.csv", BASIC_COLUMNS)print("脱敏CSV：", public_basic_path)

## 单元7：补全解析和批次循环（第5—6课时）`scrollTop` 是当前位置，`clientHeight` 是一屏高度，`scrollHeight` 是内容总高度。下一屏为 `top + height`，末尾最大为 `scrollHeight - clientHeight`。`seen_ids` 是一个集合，三批共用；`in` 判断重复，重复的直接跳过。`wait_for_function()` 在浏览器里比较新旧 ID，确认列表真的换了才继续。

In [ ]:
def get_box():    return page.locator(SCROLL_CSS)async def parse_batch(mail_ids, seen_ids, include_details=False, max_mails=0):    result = []    for mail_id in mail_ids:        if max_mails and len(seen_ids) >= max_mails:            break        if mail_id in seen_ids:       # 已在集合中，跳过（去重的关键）            continue        record = await read_list_mail(mail_id)        if record is None:            continue        if include_details:            # 第四阶段定义正文函数；本阶段False，不执行这里。            record.update(await read_detail(mail_id))        result.append(record)        seen_ids.add(mail_id)    return resultasync def collect_batches(include_details=False, max_mails=0):    await inbox.click()    await page.locator(ROW_CSS).first.wait_for()    await get_box().evaluate("box => box.scrollTop = 0")    records, seen_ids = [], set()    for batch in range(3):        top = await get_box().evaluate("box => box.scrollTop")        result = await parse_batch(await get_ids(), seen_ids, include_details, max_mails)        records.extend(result)        print(f"第{batch + 1}批，新增{len(result)}，累计{len(seen_ids)}")        if batch == 2 or (max_mails and len(seen_ids) >= max_mails):            break        box = get_box()        height = await box.evaluate("box => box.clientHeight")        full_height = await box.evaluate("box => box.scrollHeight")        next_top = min(top + height, full_height - height)        if next_top <= top:            print("已到列表末尾")            break        old_ids = await get_ids()        await box.evaluate("(box, value) => box.scrollTop = value", next_top)        # 在浏览器中比较新旧ID，改变后才继续。        await page.wait_for_function("""data => {            const ids = Array.from(document.querySelectorAll(data.selector),                                   node => node.getAttribute('data-mailid'));            return ids.length > 0 && JSON.stringify(ids) !== JSON.stringify(data.ids);        }""", arg={"selector": ROW_CSS, "ids": old_ids})    return records, seen_ids

## 单元8：执行并保存3批（第5—6课时）`nunique()` 统计唯一 ID 数，`duplicated().sum()` 统计重复 ID 数。正常结果：记录数 = 唯一 ID 数，重复 ID 数为 0。

In [ ]:
PAGE_COLUMNS = ["邮件ID"] + BASIC_COLUMNSpage_records, seen_ids = await collect_batches()page_path = save_records(page_records, "mail_pages.csv", PAGE_COLUMNS)page_df = read_records(page_path)print("CSV记录数：", len(page_df))print("唯一ID数：", page_df["邮件ID"].nunique())print("重复ID数：", page_df["邮件ID"].duplicated().sum())public_pages_path = save_public(page_records, "mail_pages_public.csv", PAGE_COLUMNS)print("脱敏CSV：", public_pages_path)

## 单元9：重复采集验证（第5—6课时）把刚才真实采过的 5 个 ID 再交给**同一个** `seen_ids` 处理。如果去重有效，新增应为 0，重新保存后 CSV 内容不变。注意：这不是关掉程序重新覆盖文件，而是在同一次运行中验证集合确实拦住了重复 ID。

In [ ]:
replay_ids = [r["邮件ID"] for r in page_records[:5]]assert replay_ids, "先成功采集一些邮件，再做重复验证"before = page_path.read_bytes()replayed = await parse_batch(replay_ids, seen_ids)print("重复采集新增：", len(replayed))save_records(page_records + replayed, "mail_pages.csv", PAGE_COLUMNS)print("CSV内容未变：", before == page_path.read_bytes())

## 单元10：补全正文函数（第7—8课时）点击主题进入详情 → 等待正文容器出现 → 读取可见文字 → `finally` 中返回列表并恢复滚动位置。`BODY_CSS` 和 `BACK_TEXT` 请按你自己的详情页核对。

In [ ]:
BODY_CSS = ".mail-content"     # 正文容器：请按详情页核对BACK_TEXT = "返回"             # “返回”控件的完整文字DOWNLOAD_ATTACHMENTS = False   # 正文必做阶段保持关闭async def read_detail(mail_id):    top = await get_box().evaluate("box => box.scrollTop")    row = get_row(mail_id)    await row.locator(SUBJECT_CSS).click()    try:        body_node = page.locator(BODY_CSS)        await body_node.wait_for(state="visible")        body = (await body_node.inner_text()).strip()        files = ""        if DOWNLOAD_ATTACHMENTS:            files = await download_attachments(mail_id)        return {"正文": body, "附件文件": files}    finally:        await page.get_by_text(BACK_TEXT, exact=True).click()        await get_box().wait_for()        await get_box().evaluate("(box, value) => box.scrollTop = value", top)        await get_row(mail_id).wait_for(state="attached")

## 单元11：先试3封（第7—8课时）`include_details=True` 才会调用正文函数，`max_mails=3` 限定只试 3 封。先确认返回、恢复位置和继续处理都正常，再做正式采集。

In [ ]:
DETAIL_COLUMNS = PAGE_COLUMNS + ["正文", "附件文件"]trial_records, _ = await collect_batches(include_details=True, max_mails=3)trial_path = save_records(trial_records, "mail_details_trial.csv", DETAIL_COLUMNS)trial_df = read_records(trial_path)print(trial_df[["邮件ID", "正文"]].assign(正文长度=trial_df["正文"].str.len()))

## 单元12：正式采集3批正文（第7—8课时）正文 CSV 共 7 列：邮件ID、发件人、邮箱、主题、时间、正文、附件文件。正文含换行时，用 pandas 的记录数统计，不要数文本行。

In [ ]:
detail_records, _ = await collect_batches(include_details=True)detail_path = save_records(detail_records, "mail_details.csv", DETAIL_COLUMNS)detail_df = read_records(detail_path)print("正文CSV记录数：", len(detail_df))print("重复ID数：", detail_df["邮件ID"].duplicated().sum())public_detail_path = save_public(detail_records, "mail_details_public.csv", DETAIL_COLUMNS)print("脱敏CSV：", public_detail_path)

## 单元13：选做附件下载（默认不执行）不选做就保持 `DOWNLOAD_ATTACHMENTS = False`，不要运行下面的调用。代码已补全，选做时先确认附件无敏感内容，再开启开关、只试 3 封，且不覆盖正式正文 CSV。

In [ ]:
ATTACH_CSS = ".attach-card"       # 附件卡片：请按详情页核对DOWNLOAD_TEXT = "下载"            # 卡片内“下载”控件文字async def download_attachments(mail_id):    cards = page.locator(ATTACH_CSS)    if await cards.count() == 0:        return ""    # 每次建新目录，避免覆盖上次的附件。    folder = PRIVATE / "attachments" / mail_id / uuid4().hex[:8]    folder.mkdir(parents=True, exist_ok=False)    files = []    for number, card in enumerate(await cards.all(), 1):        # 先监听下载，再点击。保存方法会等待下载完成。        async with page.expect_download(timeout=90000) as event:            await card.get_by_text(DOWNLOAD_TEXT, exact=True).click()        download = await event.value        name = Path(download.suggested_filename.replace("\\", "/")).name        target = folder / f"{number:02d}_{name}"        await download.save_as(target)        files.append(str(target.relative_to(PRIVATE)))    return "\n".join(files)# === 选做时才取消下面三行的注释 ===# DOWNLOAD_ATTACHMENTS = True# attachment_records, _ = await collect_batches(include_details=True, max_mails=3)# attachment_path = save_records(attachment_records, "mail_attachments_trial.csv", DETAIL_COLUMNS)# print(read_records(attachment_path)[["邮件ID", "附件文件"]])# DOWNLOAD_ATTACHMENTS = False

## 截图辅助（脱敏）把私人区域的选择器传给 `mask`，Playwright 会用色块遮住它们。提交前请目视检查生成的图片，确认没有账号、头像、邮件标题和正文。

In [ ]:
async def shot(name, mask_selectors=()):    masks = [page.locator(css) for css in mask_selectors]    await page.screenshot(        path=str(SHOTS / name),        mask=masks if masks else None,    )    print("已保存截图：", SHOTS / name)# 示例（按需取消注释）：# await shot("02_login_code.png", mask_selectors=[".cmp-account-nick", ".sidebar"])

## 下课前收尾四阶段全部完成、Notebook 已保存后，再运行下面的单元，最后双击 `stop-course-visual.bat`。下次课按顺序重新执行单元 1—12。

In [ ]:
# await browser.close()# await p.stop()print("请在确认保存后再手动执行上面两行，并运行 stop-course-visual.bat")